In [0]:
# ============================================================
# NOTEBOOK 02b — EDA ANALÍTICO + MÉTRICAS ML
# Vermont EWS V2 — Notebook de consulta paralela
#
# IMPORTANTE: Solo lectura del pipeline. No modifica
# Bronze, Trusted ni Silver del pipeline principal.
# Escribe únicamente en Silver/eda_* (carpetas nuevas).
#
# Cubre:
# SI7006 — SparkSQL, ciclo de vida, persistencia
# SI7009 — EDA, clasificación, regresión, clustering
# SI7007 — Visualizaciones analíticas
# ============================================================

# ── CELDA 1: Setup y carga de datos ────────────────────────

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings, json, os
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import (RandomForestClassifier,
                               GradientBoostingClassifier,
                               VotingClassifier)
from sklearn.preprocessing import LabelEncoder, StandardScaler, label_binarize
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, balanced_accuracy_score,
    confusion_matrix, classification_report,
    roc_curve, auc, silhouette_score
)
from sklearn.cluster import KMeans
import joblib

BRONZE  = "/Volumes/workspace/vermont/bronze"
TRUSTED = "/Volumes/workspace/vermont/trusted"
SILVER  = "/Volumes/workspace/vermont/silver"

GROUPS = [
    'Science', 'I_and_S', 'Mathematics', 'English',
    'Lengua_Castellana', 'Mandarin', 'Financial_Maths',
    'ICT_STEM', 'Physical_Education', 'Research_Methodology'
]
SUBJECT_LABELS = {
    'Science':'Science', 'I_and_S':'I&S', 'Mathematics':'Math',
    'English':'English', 'Lengua_Castellana':'Lengua',
    'Mandarin':'Mandarin', 'Financial_Maths':'Fin.Math',
    'ICT_STEM':'ICT', 'Physical_Education':'PE',
    'Research_Methodology':'Research'
}

# ── Cargar datasets ──
print("Cargando datos...")
df_train   = spark.read.parquet(f"{TRUSTED}/train_dataset_fe").toPandas()
df_predict = spark.read.parquet(f"{TRUSTED}/predict_dataset_fe").toPandas()

# Combinar para EDA interanual
df_train['year']   = '2024-25'
df_predict['year'] = '2025-26'
df_all = pd.concat([df_train, df_predict], ignore_index=True)

print(f"✓ Train   (2024-25): {len(df_train)} estudiantes, {len(df_train.columns)} columnas")
print(f"✓ Predict (2025-26): {len(df_predict)} estudiantes, {len(df_predict.columns)} columnas")
print(f"✓ Combinado:         {len(df_all)} registros")
print(f"\nColumnas disponibles (muestra):")
print([c for c in df_train.columns[:15]])

In [0]:
# ── CELDA 2: SparkSQL — Análisis con SQL ────────────────────
# Criterio SI7006: uso de SQL sobre datos en datalake

spark.createDataFrame(df_train).createOrReplaceTempView("students_24_25")
spark.createDataFrame(df_predict).createOrReplaceTempView("students_25_26")
spark.createDataFrame(df_all).createOrReplaceTempView("students_all")
print("✓ Vistas registradas en Spark Catalog")

# Q1: Distribución de riesgo por grado y año
q1 = spark.sql("""
    SELECT
        year, grade, risk_level,
        COUNT(*) as n_students,
        ROUND(COUNT(*) * 100.0 /
              SUM(COUNT(*)) OVER (PARTITION BY year, grade), 1) as pct_in_grade
    FROM students_all
    GROUP BY year, grade, risk_level
    ORDER BY year, grade,
        CASE risk_level
            WHEN 'critical'  THEN 1
            WHEN 'recovery'  THEN 2
            WHEN 'no_risk'   THEN 3
        END
""")
print("\n── Q1: Distribución de riesgo por grado y año ──")
q1.show(30, truncate=False)

# Q2: Rendimiento académico por grado
q2 = spark.sql("""
    SELECT
        year, grade,
        COUNT(*) as n_students,
        ROUND(AVG(avg_T1), 2)               as avg_T1,
        ROUND(AVG(avg_T2), 2)               as avg_T2,
        ROUND(AVG(n_bajo_T1), 2)            as avg_materias_bajo_T1,
        ROUND(AVG(n_bajo_T2), 2)            as avg_materias_bajo_T2,
        ROUND(AVG(total_absences), 1)       as avg_ausencias,
        ROUND(AVG(n_f1), 2)                 as avg_F1,
        ROUND(AVG(indice_disciplinario), 2) as avg_idx_disciplinario
    FROM students_all
    GROUP BY year, grade
    ORDER BY year, grade
""")
print("\n── Q2: Rendimiento por grado y año ──")
q2.show(truncate=False)

# Q3: Comparativa interanual global
q3 = spark.sql("""
    SELECT
        year,
        COUNT(*) as total,
        SUM(CASE WHEN risk_level='critical' THEN 1 ELSE 0 END) as criticos,
        SUM(CASE WHEN risk_level='recovery' THEN 1 ELSE 0 END) as recuperacion,
        SUM(CASE WHEN risk_level='no_risk'  THEN 1 ELSE 0 END) as sin_riesgo,
        ROUND(AVG(avg_T1), 2)         as prom_T1,
        ROUND(AVG(avg_T2), 2)         as prom_T2,
        ROUND(AVG(total_absences), 1) as prom_ausencias,
        ROUND(AVG(n_f1), 2)           as prom_F1
    FROM students_all
    GROUP BY year ORDER BY year
""")
print("\n── Q3: Comparativa interanual ──")
q3.show(truncate=False)

# Q4: Asignaturas con más estudiantes bajo mínimo
q4 = spark.sql("""
    SELECT year,
        SUM(CASE WHEN Mathematics_T2 < 4.0 THEN 1 ELSE 0 END)        as Math_bajo,
        SUM(CASE WHEN English_T2 < 4.0 THEN 1 ELSE 0 END)            as English_bajo,
        SUM(CASE WHEN Lengua_Castellana_T2 < 4.0 THEN 1 ELSE 0 END)  as Lengua_bajo,
        SUM(CASE WHEN Science_T2 < 4.0 THEN 1 ELSE 0 END)            as Science_bajo,
        SUM(CASE WHEN I_and_S_T2 < 4.0 THEN 1 ELSE 0 END)            as IandS_bajo,
        SUM(CASE WHEN Mandarin_T2 < 4.0 THEN 1 ELSE 0 END)           as Mandarin_bajo
    FROM students_all
    GROUP BY year ORDER BY year
""")
print("\n── Q4: Asignaturas con más estudiantes bajo mínimo en T2 ──")
q4.show(truncate=False)

# Guardar en Silver (carpetas nuevas — no toca el pipeline)
q1.write.mode("overwrite").parquet(f"{SILVER}/eda_riesgo_por_grado")
q2.write.mode("overwrite").parquet(f"{SILVER}/eda_rendimiento_por_grado")
q3.write.mode("overwrite").parquet(f"{SILVER}/eda_comparativa_interanual")
q4.write.mode("overwrite").parquet(f"{SILVER}/eda_asignaturas_bajo_minimo")
print("\n✓ Resultados SQL guardados en Silver/eda_*")

q1_pd = q1.toPandas()
q2_pd = q2.toPandas()
q3_pd = q3.toPandas()
q4_pd = q4.toPandas()

In [0]:
# ── CELDA 3: Visualizaciones EDA exploratorio ───────────────

colors_risk = {'critical':'#e74c3c','recovery':'#f39c12','no_risk':'#2ecc71'}
colors_year = {'2024-25':'#3498db','2025-26':'#e67e22'}

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Vermont EWS — EDA Exploratorio\n2024-25 vs 2025-26',
             fontweight='bold', fontsize=13)

# Plot 1: Distribución de riesgo por año
ax = axes[0, 0]
for i, year in enumerate(['2024-25','2025-26']):
    sub = q3_pd[q3_pd['year']==year].iloc[0]
    vals = [sub['criticos'], sub['recuperacion'], sub['sin_riesgo']]
    cats = ['Crítico','Recuperación','Sin riesgo']
    x = np.arange(len(cats))
    ax.bar(x + i*0.35, vals, width=0.35,
           color=colors_year[year], label=year, alpha=0.85)
    for j, v in enumerate(vals):
        ax.text(j + i*0.35, v + 0.3, str(int(v)),
                ha='center', fontsize=9, fontweight='bold')
ax.set_xticks(x + 0.175)
ax.set_xticklabels(cats)
ax.set_title('Distribución de riesgo por año', fontweight='bold')
ax.set_ylabel('Estudiantes')
ax.legend()
ax.grid(alpha=0.2)

# Plot 2: Promedio T1 vs T2 por grado
ax = axes[0, 1]
for year in ['2024-25','2025-26']:
    sub = q2_pd[q2_pd['year']==year].sort_values('grade')
    ax.plot(sub['grade'].astype(str), sub['avg_T1'], 'o--',
            color=colors_year[year], label=f'{year} T1', alpha=0.7)
    ax.plot(sub['grade'].astype(str), sub['avg_T2'], 's-',
            color=colors_year[year], label=f'{year} T2', linewidth=2)
ax.axhline(y=4.0, color='red', linestyle=':', alpha=0.5, label='Mínimo 4.0')
ax.set_xlabel('Grado')
ax.set_ylabel('Promedio')
ax.set_title('Promedio T1 vs T2 por grado', fontweight='bold')
ax.legend(fontsize=7)
ax.grid(alpha=0.2)

# Plot 3: Materias bajo mínimo en T2
ax = axes[0, 2]
for year in ['2024-25','2025-26']:
    sub = q2_pd[q2_pd['year']==year].sort_values('grade')
    ax.bar(sub['grade'].astype(str) + f'\n{year[-2:]}',
           sub['avg_materias_bajo_T2'],
           color=colors_year[year], alpha=0.8, label=year)
ax.axhline(y=3, color='red', linestyle='--', alpha=0.5,
           label='3 mat. = pierde año')
ax.set_title('Promedio materias bajo 4.0 en T2', fontweight='bold')
ax.set_ylabel('N° materias')
ax.legend(fontsize=8)
ax.grid(alpha=0.2)

# Plot 4: Ausencias por nivel de riesgo
ax = axes[1, 0]
df_abs = df_all.groupby(['year','risk_level'])['total_absences'].mean().reset_index()
risks = ['critical','recovery','no_risk']
labels_r = ['Crítico','Recuperación','Sin riesgo']
x = np.arange(len(risks))
for i, year in enumerate(['2024-25','2025-26']):
    sub = df_abs[df_abs['year']==year]
    vals = [sub[sub['risk_level']==r]['total_absences'].values[0]
            if len(sub[sub['risk_level']==r]) > 0 else 0 for r in risks]
    ax.bar(x + i*0.35, vals, width=0.35,
           color=colors_year[year], label=year, alpha=0.85)
ax.set_xticks(x + 0.175)
ax.set_xticklabels(labels_r)
ax.set_title('Ausencias promedio por nivel de riesgo', fontweight='bold')
ax.set_ylabel('Ausencias')
ax.legend()
ax.grid(alpha=0.2)

# Plot 5: Índice disciplinario por riesgo
ax = axes[1, 1]
df_disc = df_all.groupby(['year','risk_level'])['indice_disciplinario'].mean().reset_index()
for i, year in enumerate(['2024-25','2025-26']):
    sub = df_disc[df_disc['year']==year]
    vals = [sub[sub['risk_level']==r]['indice_disciplinario'].values[0]
            if len(sub[sub['risk_level']==r]) > 0 else 0 for r in risks]
    ax.bar(x + i*0.35, vals, width=0.35,
           color=colors_year[year], label=year, alpha=0.85)
ax.set_xticks(x + 0.175)
ax.set_xticklabels(labels_r)
ax.set_title('Índice disciplinario por nivel de riesgo\n(F1×1 + F2×3)',
             fontweight='bold')
ax.set_ylabel('Índice')
ax.legend()
ax.grid(alpha=0.2)

# Plot 6: Dispersión avg_T1 vs avg_T2 por riesgo (2025-26)
ax = axes[1, 2]
for risk, color in colors_risk.items():
    sub = df_predict[df_predict['risk_level']==risk]
    label_map = {'critical':'Crítico','recovery':'Recuperación','no_risk':'Sin riesgo'}
    ax.scatter(sub['avg_T1'], sub['avg_T2'],
               c=color, label=label_map[risk], alpha=0.6, s=50)
ax.axhline(y=4.0, color='red', linestyle='--', alpha=0.4)
ax.axvline(x=4.0, color='red', linestyle='--', alpha=0.4)
ax.set_xlabel('Promedio T1')
ax.set_ylabel('Promedio T2')
ax.set_title('T1 vs T2 por nivel de riesgo\n(2025-26)', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('/tmp/eda_exploratorio.png', dpi=150, bbox_inches='tight')
display(fig)
plt.close()
print("✓ EDA exploratorio completo")

In [0]:
# ── CELDA 4a: Árbol de decisión — overfitting vs. podado ───
# Contexto metodológico: así llegamos al Random Forest
# Criterio SI7009: demostración de overfitting

print("=" * 60)
print("ÁRBOL DE DECISIÓN — Overfitting vs. Podado")
print("Contexto metodológico: motivación del Random Forest")
print("=" * 60)

# Preparar datos
FEATURES_CLASIF = (
    [f'{g}_T1' for g in GROUPS] +
    [f'{g}_T2' for g in GROUPS] +
    [f'{g}_delta' for g in GROUPS if f'{g}_delta' in df_train.columns] +
    ['total_absences', 'late', 'n_f1', 'n_f2',
     'tendencia_general', 'avg_T1', 'avg_T2',
     'n_bajo_T1', 'n_bajo_T2', 'delta_materias_bajo',
     'dispersion_T2', 'indice_disciplinario',
     'min_nota_T2', 'min_nota_T1']
)
available_clasif = [f for f in FEATURES_CLASIF if f in df_train.columns]

X_train = df_train[available_clasif].fillna(df_train[available_clasif].mean())
y_train = df_train['risk_level']
X_eval  = df_predict[available_clasif].fillna(df_train[available_clasif].mean())
y_eval  = df_predict['risk_level']

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_eval_enc  = le.transform(y_eval)
classes     = le.classes_

print(f"\n✓ Features: {len(available_clasif)}")
print(f"✓ Train: {len(X_train)} | Eval: {len(X_eval)}")
print(f"✓ Clases: {list(classes)}")

# Árbol SIN podar — sobreajustado
dt_overfit = DecisionTreeClassifier(random_state=42)
dt_overfit.fit(X_train, y_train_enc)
acc_train_over = accuracy_score(y_train_enc, dt_overfit.predict(X_train))
acc_eval_over  = accuracy_score(y_eval_enc, dt_overfit.predict(X_eval))
f1_over        = f1_score(y_eval_enc, dt_overfit.predict(X_eval), average='macro')

print(f"\n── Árbol sin podar (max_depth=None) ──")
print(f"  Profundidad: {dt_overfit.get_depth()}")
print(f"  Hojas:       {dt_overfit.get_n_leaves()}")
print(f"  Acc train:   {acc_train_over:.3f} ← sobreajuste")
print(f"  Acc eval:    {acc_eval_over:.3f}")
print(f"  F1 macro:    {f1_over:.3f}")

# Árbol PODADO — generaliza mejor
dt_podado = DecisionTreeClassifier(max_depth=4, min_samples_leaf=5, random_state=42)
dt_podado.fit(X_train, y_train_enc)
acc_train_pod = accuracy_score(y_train_enc, dt_podado.predict(X_train))
acc_eval_pod  = accuracy_score(y_eval_enc, dt_podado.predict(X_eval))
f1_pod        = f1_score(y_eval_enc, dt_podado.predict(X_eval), average='macro')

print(f"\n── Árbol podado (max_depth=4, min_samples_leaf=5) ──")
print(f"  Profundidad: {dt_podado.get_depth()}")
print(f"  Hojas:       {dt_podado.get_n_leaves()}")
print(f"  Acc train:   {acc_train_pod:.3f}")
print(f"  Acc eval:    {acc_eval_pod:.3f}")
print(f"  F1 macro:    {f1_pod:.3f}")

# Visualización comparativa
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('Árbol de Decisión — Overfitting vs. Podado\nMotivación del Random Forest',
             fontweight='bold', fontsize=13)

# Plot 1: Comparativa de métricas
ax = axes[0]
metricas = ['Acc Train','Acc Eval','F1 Macro']
vals_over = [acc_train_over, acc_eval_over, f1_over]
vals_pod  = [acc_train_pod,  acc_eval_pod,  f1_pod]
x = np.arange(len(metricas))
ax.bar(x - 0.2, vals_over, width=0.35, color='#e74c3c',
       label='Sin podar (overfitting)', alpha=0.85)
ax.bar(x + 0.2, vals_pod,  width=0.35, color='#27ae60',
       label='Podado (max_depth=4)', alpha=0.85)
for i, (vo, vp) in enumerate(zip(vals_over, vals_pod)):
    ax.text(i - 0.2, vo + 0.01, f'{vo:.2f}', ha='center', fontsize=9)
    ax.text(i + 0.2, vp + 0.01, f'{vp:.2f}', ha='center', fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(metricas)
ax.set_ylim(0, 1.15)
ax.set_title('Comparativa de métricas', fontweight='bold')
ax.set_ylabel('Score')
ax.legend(fontsize=9)
ax.grid(alpha=0.2)
ax.axhline(y=1.0, color='gray', linestyle=':', alpha=0.5)

# Anotación de overfitting
ax.annotate('Overfitting:\nacc_train=1.0\nacc_eval baja',
            xy=(0 - 0.2, acc_train_over),
            xytext=(0.5, 0.95),
            arrowprops=dict(arrowstyle='->', color='red'),
            fontsize=8, color='red')

# Plot 2: Árbol podado visualizado
ax = axes[1]
plot_tree(dt_podado, ax=ax,
          feature_names=available_clasif,
          class_names=list(classes),
          filled=True, rounded=True,
          fontsize=6, max_depth=3)
ax.set_title('Árbol podado (max_depth=4)\nInterpretable pero limitado',
             fontweight='bold')

# Plot 3: Narrativa hacia RF
ax = axes[2]
ax.axis('off')
texto = """
¿Por qué Random Forest?

El árbol de decisión sin podar
memoriza el conjunto de entrenamiento
(acc_train = 1.0) pero generaliza
mal a nuevos datos → OVERFITTING.

El árbol podado mejora la
generalización pero sacrifica
capacidad predictiva.

Random Forest resuelve esto:
1. Entrena N árboles en subconjuntos
   aleatorios de datos (Bagging)
2. Cada árbol usa subconjunto
   aleatorio de features
3. La predicción final es el voto
   mayoritario de todos los árboles

Resultado: menor varianza,
mejor generalización, sin perder
capacidad de capturar no-linealidades.

Vermont EWS usa RF con:
- 5-Fold CV estratificado
- Grid Search de hiperparámetros
- Class weights por desbalanceo
- Umbral optimizado (0.30)
"""
ax.text(0.05, 0.95, texto, transform=ax.transAxes,
        fontsize=10, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='#eaf4fb', alpha=0.8))
ax.set_title('Motivación del Random Forest', fontweight='bold')

plt.tight_layout()
plt.savefig('/tmp/arbol_overfitting.png', dpi=150, bbox_inches='tight')
display(fig)
plt.close()
print("\n✓ Árbol de decisión — análisis completo")

In [0]:
# ── CELDA 4b: Métricas reales del clasificador ──────────────
# Carga desde model_metadata.json — números del pipeline real
# NO reentrena nada

print("=" * 60)
print("CLASIFICADOR — Métricas del modelo en producción")
print("Fuente: Silver/models/model_metadata.json")
print("=" * 60)

import json
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

SILVER = "/Volumes/workspace/vermont/silver"

with open(f"{SILVER}/models/model_metadata.json") as f:
    meta_clf = json.load(f)

print(f"\n── Modelo: {meta_clf['modelo']} {meta_clf['version']} ──")
print(f"  Entrenado:    {meta_clf['fecha_entrenamiento']}")
print(f"  Dataset:      {meta_clf['dataset_train']}")
print(f"  Estudiantes:  {meta_clf['n_estudiantes_train']}")
print(f"  Features:     {meta_clf['n_features']}")
print(f"  Umbral:       {meta_clf['umbral_produccion']}")
print(f"\n── Hiperparámetros ──")
for k, v in meta_clf['hiperparametros'].items():
    print(f"  {k}: {v}")
print(f"\n── Métricas CV (5-Fold estratificado) ──")
for k, v in meta_clf['metricas_cv'].items():
    print(f"  {k}: {v}")
print(f"\n── Justificación del umbral ──")
print(f"  {meta_clf['justificacion_umbral']}")
print(f"\n── Anti-leakage ──")
for item in meta_clf['anti_leakage']:
    print(f"  ✓ {item}")

# Tabla comparativa de 5 modelos (del ejercicio 02b — académico)
# Los del CV son del ejercicio — el modelo real es el de metadata
print("\n── Comparativa 5 modelos (ejercicio académico 02b) ──")
df_comp = pd.DataFrame([
    {"Modelo": "Regresión Logística", "Accuracy": 0.657,
     "Balanced Acc": 0.557, "F1 macro": 0.523},
    {"Modelo": "Árbol de Decisión",   "Accuracy": 0.555,
     "Balanced Acc": 0.510, "F1 macro": 0.471},
    {"Modelo": "Random Forest ✅",    "Accuracy": 0.762,
     "Balanced Acc": 0.610, "F1 macro": 0.608},
    {"Modelo": "Gradient Boosting",   "Accuracy": 0.710,
     "Balanced Acc": 0.601, "F1 macro": 0.576},
    {"Modelo": "Voting Ensemble",     "Accuracy": 0.735,
     "Balanced Acc": 0.596, "F1 macro": 0.578},
])
print(df_comp.to_string(index=False))
print("  Nota: comparativa con sklearn para ejercicio metodológico.")
print("  El modelo en producción usa SparkML con hiperparámetros optimizados.")

# Métricas finales del modelo real
print(f"\n── Modelo seleccionado — RF pipeline (umbral 0.30) ──")
print(f"  F1 macro CV:      {meta_clf['metricas_cv']['f1_macro']}")
print(f"  Recall critical:  {meta_clf['metricas_cv']['recall_critical']}")
print(f"  F2 critical:      {meta_clf['metricas_cv']['f2_critical']}")

In [0]:
# ── CELDA 4c: Visualizaciones clasificador ──────────────────
# Matriz de confusión y ROC desde métricas reales del pipeline

import joblib
import numpy as np
from sklearn.metrics import confusion_matrix, roc_curve, auc

print("Generando visualizaciones del clasificador...")

# Cargar modelo y datos para generar matriz y ROC reales
TRUSTED = "/Volumes/workspace/vermont/trusted"

try:
    rf_clf  = joblib.load(f"{SILVER}/models/rf_classifier.joblib")
    imp_clf = joblib.load(f"{SILVER}/models/imputer.joblib")
    le_clf  = joblib.load(f"{SILVER}/models/label_encoder.joblib")
    with open(f"{SILVER}/models/feature_list.json") as f:
        feats_clf = json.load(f)

    df_pred_raw = spark.read.parquet(
        f"{TRUSTED}/predict_dataset_fe"
    ).toPandas()

    avail = [f for f in feats_clf if f in df_pred_raw.columns]
    X_eval = imp_clf.transform(df_pred_raw[avail].fillna(0))
    probas = rf_clf.predict_proba(X_eval)
    classes = le_clf.classes_
    idx_crit = list(classes).index('critical')
    idx_rec  = list(classes).index('recovery')
    idx_norisk = list(classes).index('no_risk')

    # Umbral 0.30
    y_pred = []
    for p in probas:
        if p[idx_crit] >= 0.30:
            y_pred.append('critical')
        else:
            resto = p.copy()
            resto[idx_crit] = -1
            y_pred.append(classes[np.argmax(resto)])
    y_pred = np.array(y_pred)
    y_true = df_pred_raw['risk_level'].values

    label_order = ['critical', 'recovery', 'no_risk']
    labels_es   = ['Crítico', 'Recuperación', 'Sin riesgo']
    cm = confusion_matrix(y_true, y_pred, labels=label_order)
    cm_pct = cm.astype(float)
    for i in range(len(cm)):
        if cm[i].sum() > 0:
            cm_pct[i] = cm[i] / cm[i].sum() * 100

    # AUC por clase
    auc_scores = {}
    fpr_dict   = {}
    tpr_dict   = {}
    for cls in label_order:
        y_bin   = (y_true == cls).astype(int)
        cls_idx = list(classes).index(cls)
        fpr, tpr, _ = roc_curve(y_bin, probas[:, cls_idx])
        auc_scores[cls] = round(auc(fpr, tpr), 3)
        fpr_dict[cls]   = fpr
        tpr_dict[cls]   = tpr

    modelo_ok = True
    print("✓ Modelo cargado — métricas reales del pipeline")

except Exception as e:
    print(f"⚠ Error: {e}")
    modelo_ok = False

# Visualizaciones
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle(
    'Vermont EWS — Clasificador de Riesgo\n'
    'Random Forest · Umbral 0.30 · '
    f'F1 macro CV = {meta_clf["metricas_cv"]["f1_macro"]} · '
    f'Recall critical CV = {meta_clf["metricas_cv"]["recall_critical"]}',
    fontweight='bold', fontsize=12
)

# Plot 1: Comparativa 5 modelos
ax = axes[0]
modelos_n = df_comp['Modelo']
metricas_p = ['Accuracy', 'Balanced Acc', 'F1 macro']
x = np.arange(len(modelos_n))
colors_m = ['#3498db', '#e67e22', '#2ecc71']
for i, met in enumerate(metricas_p):
    ax.bar(x + i*0.25, df_comp[met], width=0.22,
           color=colors_m[i], label=met, alpha=0.85)
ax.set_xticks(x + 0.25)
ax.set_xticklabels(modelos_n, rotation=30, ha='right', fontsize=7)
ax.set_ylim(0, 1.1)
ax.set_title('Comparativa 5 modelos\n(sklearn · ejercicio académico)',
             fontweight='bold')
ax.set_ylabel('Score')
ax.legend(fontsize=8)
ax.grid(alpha=0.2)

# Plot 2: Matriz de confusión
ax = axes[1]
if modelo_ok:
    im = ax.imshow(cm_pct, cmap='Blues', vmin=0, vmax=100)
    ax.set_xticks(range(3))
    ax.set_yticks(range(3))
    ax.set_xticklabels(
        [f'Pred:\n{l}' for l in labels_es], fontsize=9)
    ax.set_yticklabels(
        [f'Real: {l}' for l in labels_es], fontsize=9)
    for i in range(3):
        for j in range(3):
            ax.text(j, i,
                    f'{cm[i,j]}\n({cm_pct[i,j]:.0f}%)',
                    ha='center', va='center', fontsize=10,
                    color='white' if cm_pct[i,j] > 50 else 'black',
                    fontweight='bold')
    ax.set_title(
        f'Matriz de confusión real\n'
        f'Recall critical = {meta_clf["metricas_cv"]["recall_critical"]}',
        fontweight='bold')
    plt.colorbar(im, ax=ax)
else:
    ax.text(0.5, 0.5, 'Modelo no disponible',
            ha='center', va='center', transform=ax.transAxes)

# Plot 3: Curva ROC
ax = axes[2]
colors_roc = {
    'critical': '#e74c3c',
    'recovery': '#f39c12',
    'no_risk':  '#2ecc71'
}
label_map_es = {
    'critical': 'Crítico',
    'recovery': 'Recuperación',
    'no_risk':  'Sin riesgo'
}
if modelo_ok:
    for cls in label_order:
        ax.plot(fpr_dict[cls], tpr_dict[cls],
                color=colors_roc[cls], linewidth=2.5,
                label=f"{label_map_es[cls]} "
                      f"(AUC={auc_scores[cls]})")
ax.plot([0,1],[0,1], '--', color='gray',
        linewidth=1.5, label='Aleatorio')
auc_macro = round(np.mean(list(auc_scores.values())), 3) \
            if modelo_ok else '—'
ax.set_xlabel('Tasa Falsos Positivos (FPR)')
ax.set_ylabel('Tasa Verdaderos Positivos (TPR)')
ax.set_title(f'Curva ROC — One vs Rest\nAUC macro = {auc_macro}',
             fontweight='bold')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/clasificacion_real.png', dpi=150, bbox_inches='tight')
display(fig)
plt.close()
print("✓ Visualizaciones clasificador completas")

In [0]:
# ── CELDA 4d: Métricas reales del regresor ──────────────────
# Carga desde reg_metadata.json — números del pipeline real

print("=" * 60)
print("REGRESOR T3 — Métricas del modelo en producción")
print("Fuente: Silver/models/reg_metadata.json")
print("=" * 60)

with open(f"{SILVER}/models/reg_metadata.json") as f:
    meta_reg = json.load(f)

print(f"\n── Modelo: {meta_reg['modelo']} ──")
print(f"  Tipo:         {meta_reg['tipo']}")
print(f"  Entrenado:    {meta_reg['fecha_entrenamiento']}")
print(f"  Features:     {meta_reg['n_features']}")
print(f"  Targets:      {meta_reg['n_targets']} materias")
print(f"\n── Hiperparámetros ──")
for k, v in meta_reg['hiperparametros'].items():
    print(f"  {k}: {v}")
print(f"\n── Métricas globales CV ──")
for k, v in meta_reg['metricas_cv'].items():
    print(f"  {k}: {v}")
print(f"\n── Incertidumbre ──")
inc = meta_reg['incertidumbre']
print(f"  Método:            {inc['metodo']}")
print(f"  Amplitud promedio: {inc['amplitud_promedio']}")
print(f"  Interpretación:    {inc['interpretacion']}")

# Tabla por materia
print("\n── Confiabilidad por materia ──")
SUBJECT_LABELS = {
    'Science':'Science', 'I_and_S':'I&S',
    'Mathematics':'Math', 'English':'English',
    'Lengua_Castellana':'Lengua', 'Mandarin':'Mandarin',
    'Financial_Maths':'Fin.Math', 'ICT_STEM':'ICT',
    'Physical_Education':'PE', 'Research_Methodology':'Research'
}
reg_rows = []
for mat, info in meta_reg['confiabilidad_por_materia'].items():
    reg_rows.append({
        'Materia':     SUBJECT_LABELS.get(mat, mat),
        'R² CV':       info['r2_cv'],
        'MAE CV':      info['mae_cv'],
        'Confiable':   info['confiable'],
        'Amp P10-P90': info['amplitud_p10_p90']
    })
df_reg = pd.DataFrame(reg_rows).sort_values('R² CV', ascending=False)
print(df_reg.to_string(index=False))

# Visualización regresor
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(
    'Vermont EWS — Regresor T3 por Materia\n'
    f'Random Forest Multi-output · '
    f'MAE macro={meta_reg["metricas_cv"]["MAE_macro"]} · '
    f'R² macro={meta_reg["metricas_cv"]["R2_macro"]}',
    fontweight='bold', fontsize=12
)

# Plot 1: R² por materia
ax = axes[0]
colors_r2 = ['#27ae60' if r >= 0.25
              else '#f39c12' if r >= 0
              else '#e74c3c'
              for r in df_reg['R² CV']]
bars = ax.barh(df_reg['Materia'], df_reg['R² CV'],
               color=colors_r2, alpha=0.85)
ax.axvline(x=0, color='black', linewidth=0.8)
ax.axvline(x=meta_reg['metricas_cv']['R2_macro'],
           color='#3498db', linestyle='--', alpha=0.7,
           label=f'R² macro ({meta_reg["metricas_cv"]["R2_macro"]})')
for i, (mat, r2) in enumerate(zip(df_reg['Materia'],
                                   df_reg['R² CV'])):
    ax.text(r2 + 0.01 if r2 >= 0 else r2 - 0.01,
            i, f'{r2:.3f}', va='center', fontsize=9,
            ha='left' if r2 >= 0 else 'right')

# Marcar confiabilidad
for i, (_, row) in enumerate(df_reg.iterrows()):
    icon = '✅' if row['Confiable'] == 'confiable' \
           else '⚠️' if row['Confiable'] == 'orientativo' \
           else '❌'
    ax.text(ax.get_xlim()[1]*0.98, i, icon,
            va='center', ha='right', fontsize=10)

ax.set_xlabel('R²')
ax.set_title('R² por materia\n✅confiable ⚠️orientativo ❌no viable',
             fontweight='bold')
ax.legend(fontsize=8)
ax.grid(alpha=0.2)

# Plot 2: MAE + amplitud P10-P90
ax = axes[1]
x = np.arange(len(df_reg))
ax.bar(x - 0.2, df_reg['MAE CV'], width=0.35,
       color='#3498db', label='MAE CV', alpha=0.85)
ax.bar(x + 0.2, df_reg['Amp P10-P90'], width=0.35,
       color='#e67e22', label='Amplitud P10-P90', alpha=0.85)
ax.axhline(y=meta_reg['metricas_cv']['MAE_macro'],
           color='#3498db', linestyle='--', alpha=0.6,
           label=f'MAE macro ({meta_reg["metricas_cv"]["MAE_macro"]})')
ax.set_xticks(x)
ax.set_xticklabels(df_reg['Materia'], rotation=30,
                   ha='right', fontsize=8)
ax.set_ylabel('Escala 1-7')
ax.set_title('MAE y amplitud intervalo P10-P90\npor materia',
             fontweight='bold')
ax.legend(fontsize=8)
ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('/tmp/regresion_real.png', dpi=150, bbox_inches='tight')
display(fig)
plt.close()
print("\n✓ Métricas del regresor completas")
print(f"  Uso recomendado: {meta_reg['uso_recomendado']}")

In [0]:
# ── CELDA 4e: Categorías de alerta — desde pipeline real ───
# Carga df_pred desde Silver — misma lógica que 03_apply_model
# NO recalcula — usa las categorías ya calculadas por el pipeline

print("=" * 60)
print("CATEGORÍAS DE ALERTA — desde pipeline real")
print("Fuente: Silver/predictions_25_26_v2")
print("=" * 60)

# Cargar predicciones reales del pipeline
df_pred_cats = spark.read.parquet(
    f"{SILVER}/predictions_25_26_v2"
).toPandas()

# Cargar early_alerts que tiene categoria ya calculada
df_alerts = spark.read.parquet(
    f"{SILVER}/early_alerts_v2"
).toPandas()

# Unir para tener proba_critical + categoria + n_bajo_acumulada
cols_pred = ['student_id','proba_critical','proba_recovery',
             'proba_riesgo','pred_label','confianza']
cols_alerts = ['student_id','categoria','n_bajo_acumulada',
               't3_confirma_riesgo','grade','section_anon']

df_cats = df_alerts[cols_alerts].merge(
    df_pred_cats[[c for c in cols_pred
                  if c in df_pred_cats.columns]],
    on='student_id', how='left'
)

# Si proba_critical no está en predictions, buscar en alerts
if 'proba_critical' not in df_cats.columns:
    if 'proba_critical' in df_alerts.columns:
        df_cats['proba_critical'] = df_alerts['proba_critical'].values

print(f"✓ {len(df_cats)} estudiantes cargados")

# ── Distribución ──
ALERT_ORDER = ['Riesgo Confirmado','Punto Ciego',
               'Riesgo Teórico','Sin Riesgo']
print("\n── Distribución de categorías (pipeline real) ──")
for cat in ALERT_ORDER:
    n   = (df_cats['categoria'] == cat).sum()
    pct = round(n / len(df_cats) * 100, 1)
    print(f"  {cat:22s}: {n:3d} ({pct}%)")

# ── Cruce categoría × risk_level real ──
if 'risk_level' in df_cats.columns:
    print("\n── Cruce categoría × risk_level real ──")
    cross = pd.crosstab(df_cats['categoria'],
                        df_cats['risk_level'])
    print(cross.to_string())

# ── Visualizaciones ──
ALERT_COLORS = {
    'Riesgo Confirmado': '#e74c3c',
    'Punto Ciego':       '#e67e22',
    'Riesgo Teórico':    '#3498db',
    'Sin Riesgo':        '#2ecc71',
}
ALERT_EMOJI = {
    'Riesgo Confirmado': '🔴',
    'Punto Ciego':       '🟠',
    'Riesgo Teórico':    '🔵',
    'Sin Riesgo':        '🟢',
}

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle(
    'Vermont EWS — 4 Categorías de Alerta\n'
    'Fuente: pipeline real (03_apply_model + 04_early_alert) · 2025-26',
    fontweight='bold', fontsize=13
)

# Plot 1: Scatter proba_critical vs n_bajo_acumulada
ax = axes[0]
for cat in ALERT_ORDER:
    sub = df_cats[df_cats['categoria'] == cat]
    if sub.empty:
        continue
    ax.scatter(
        sub['proba_critical'],
        sub['n_bajo_acumulada'],
        c=ALERT_COLORS[cat],
        label=f"{ALERT_EMOJI[cat]} {cat} (n={len(sub)})",
        alpha=0.75, s=60
    )

ax.axvline(x=0.30, color='#e74c3c', linestyle='--',
           linewidth=1.5, alpha=0.7,
           label='Umbral modelo (0.30)')
ax.axhline(y=2.5, color='#888', linestyle='--',
           linewidth=1.5, alpha=0.7,
           label='T3 confirma riesgo (≥3 materias)')

# Anotaciones de cuadrantes
ymax = df_cats['n_bajo_acumulada'].max()
ax.text(0.02, ymax * 0.95, '🟠 Punto Ciego',
        fontsize=8, color='#e67e22', alpha=0.8)
ax.text(0.55, ymax * 0.95, '🔴 Riesgo Confirmado',
        fontsize=8, color='#e74c3c', alpha=0.8)
ax.text(0.02, 0.15, '🟢 Sin Riesgo',
        fontsize=8, color='#27ae60', alpha=0.8)
ax.text(0.55, 0.15, '🔵 Riesgo Teórico',
        fontsize=8, color='#3498db', alpha=0.8)

ax.set_xlabel('Probabilidad de riesgo (modelo)')
ax.set_ylabel('N° materias en riesgo acumulado')
ax.set_title('Mapa de riesgo por cuadrante\n(igual al semáforo del dashboard)',
             fontweight='bold')
ax.legend(fontsize=8, loc='upper left')
ax.grid(alpha=0.2)

# Plot 2: Distribución de categorías
ax = axes[1]
cats_counts = [(df_cats['categoria'] == cat).sum()
               for cat in ALERT_ORDER]
bars = ax.bar(
    [f"{ALERT_EMOJI[c]}\n{c}" for c in ALERT_ORDER],
    cats_counts,
    color=[ALERT_COLORS[c] for c in ALERT_ORDER],
    alpha=0.85
)
for bar, n in zip(bars, cats_counts):
    pct = round(n / len(df_cats) * 100, 1)
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.3,
            f'{n}\n({pct}%)',
            ha='center', fontsize=9, fontweight='bold')
ax.set_title('Distribución de categorías\n2025-26',
             fontweight='bold')
ax.set_ylabel('Estudiantes')
ax.tick_params(axis='x', labelsize=8)
ax.grid(alpha=0.2)

# Plot 3: Cruce categoría × n_bajo_acumulada (boxplot)
ax = axes[2]
data_box = [
    df_cats[df_cats['categoria'] == cat]['n_bajo_acumulada'].values
    for cat in ALERT_ORDER
]
bp = ax.boxplot(data_box, patch_artist=True,
                labels=[f"{ALERT_EMOJI[c]}\n{c.split()[0]}"
                        for c in ALERT_ORDER])
for patch, cat in zip(bp['boxes'], ALERT_ORDER):
    patch.set_facecolor(ALERT_COLORS[cat])
    patch.set_alpha(0.7)
ax.axhline(y=2.5, color='red', linestyle='--',
           alpha=0.5, label='Umbral ≥3 materias')
ax.set_ylabel('N° materias en riesgo acumulado')
ax.set_title('Distribución materias en riesgo\npor categoría',
             fontweight='bold')
ax.legend(fontsize=8)
ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('/tmp/categorias_alerta_real.png',
            dpi=150, bbox_inches='tight')
display(fig)
plt.close()

print("\n✓ Categorías de alerta cargadas desde pipeline real")
print("  Lógica idéntica al dashboard — sin recalcular")

In [0]:
# ── CELDA 5: Clustering K-Means — Elbow + Silhouette ───────
# Criterio SI7009: clustering no supervisado, selección de K

print("=" * 60)
print("CLUSTERING K-MEANS — Selección de K y perfiles")
print(f"Vermont School 2025-26 | {len(df_predict)} estudiantes")
print("=" * 60)

FEATURES_CLUSTER = [
    'avg_T1', 'avg_T2', 'tendencia_general',
    'n_bajo_T1', 'n_bajo_T2', 'dispersion_T2',
    'min_nota_T2', 'n_destacadas_T2',
    'total_absences', 'ratio_ausencia_clase',
    'n_f1', 'n_f2', 'indice_disciplinario',
]
available_cl = [f for f in FEATURES_CLUSTER if f in df_predict.columns]
print(f"✓ Features: {len(available_cl)}")

X_cl     = df_predict[available_cl].fillna(df_predict[available_cl].mean())
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_cl)

# Elbow + Silhouette
inercias    = []
silhouettes = []
K_range     = range(2, 10)

print("\n── Elbow Method ──")
for k in K_range:
    km  = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
    km.fit(X_scaled)
    inercias.append(km.inertia_)
    sil = silhouette_score(X_scaled, km.labels_)
    silhouettes.append(sil)
    print(f"  K={k}: inercia={km.inertia_:.1f} | silhouette={sil:.3f}")

K_OPTIMO = list(K_range)[silhouettes.index(max(silhouettes))]
print(f"\n✓ K óptimo: {K_OPTIMO} (silhouette={max(silhouettes):.3f})")
print(f"  Nota: el pipeline usa K=2 por consistencia entre ejecuciones")

# K-Means con K=2 (igual al pipeline)
km_final = KMeans(n_clusters=2, random_state=42, n_init=10, max_iter=300)
clusters = km_final.fit_predict(X_scaled)
df_predict_cl = df_predict.copy()
df_predict_cl['cluster'] = clusters

# Nombrar perfiles por avg_T2
avg_global = df_predict['avg_T2'].mean()
nombres    = {}
for k in range(2):
    mask  = clusters == k
    avg_k = df_predict.loc[mask, 'avg_T2'].mean()
    nombres[k] = 'Rendimiento sólido' if avg_k >= avg_global \
                 else 'Riesgo multidimensional'
df_predict_cl['perfil'] = df_predict_cl['cluster'].map(nombres)

print("\n── Distribución de perfiles ──")
for perfil, sub in df_predict_cl.groupby('perfil'):
    print(f"  {perfil}: {len(sub)} estudiantes ({len(sub)/len(df_predict_cl)*100:.1f}%)")

# Cruce con categorías de alerta
print("\n── Cruce perfil × nivel de riesgo ──")
cross = pd.crosstab(df_predict_cl['perfil'], df_predict_cl['risk_level'])
print(cross.to_string())

# Visualizaciones
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Vermont EWS — Clustering K-Means\nPerfiles de estudiantes 2025-26',
             fontweight='bold', fontsize=13)

colors_cl = ['#e74c3c','#3498db']

# Plot 1: Elbow
ax = axes[0, 0]
ax.plot(list(K_range), inercias, 'o-', color='#e74c3c', linewidth=2)
ax.axvline(x=K_OPTIMO, color='gray', linestyle='--', alpha=0.7,
           label=f'K={K_OPTIMO} óptimo (silhouette)')
ax.axvline(x=2, color='blue', linestyle=':', alpha=0.7,
           label='K=2 usado en pipeline')
ax.set_xlabel('K')
ax.set_ylabel('Inercia')
ax.set_title('Elbow Method', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# Plot 2: Silhouette
ax = axes[0, 1]
ax.plot(list(K_range), silhouettes, 's-', color='#3498db', linewidth=2)
ax.axvline(x=K_OPTIMO, color='gray', linestyle='--', alpha=0.7,
           label=f'K={K_OPTIMO} (sil={max(silhouettes):.3f})')
ax.set_xlabel('K')
ax.set_ylabel('Silhouette Score')
ax.set_title('Silhouette Score (mayor = mejor)', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# Plot 3: Scatter T1 vs T2 por perfil
ax = axes[1, 0]
for k in range(2):
    sub     = df_predict_cl[df_predict_cl['cluster']==k]
    perfil_k = nombres[k]
    ax.scatter(sub['avg_T1'], sub['avg_T2'],
               c=colors_cl[k], label=perfil_k, alpha=0.7, s=60)
ax.axhline(y=4.0, color='red', linestyle='--', alpha=0.4)
ax.axvline(x=4.0, color='red', linestyle='--', alpha=0.4)
ax.set_xlabel('Promedio T1')
ax.set_ylabel('Promedio T2')
ax.set_title('Clusters: T1 vs T2', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(alpha=0.2)

# Plot 4: Características por perfil (barras comparativas)
ax = axes[1, 1]
feat_radar = ['avg_T2','n_bajo_T2','total_absences',
              'indice_disciplinario','dispersion_T2']
labels_r   = ['Prom T2','Mat<4 T2','Ausencias','Idx Disc','Dispersión']
perfil_means = df_predict_cl.groupby('perfil')[feat_radar].mean()

x = np.arange(len(feat_radar))
for i, (perfil, row) in enumerate(perfil_means.iterrows()):
    vals_norm = (row - row.min()) / (row.max() - row.min() + 1e-6)
    ax.bar(x + i*0.35, vals_norm, width=0.3,
           color=colors_cl[i], label=perfil, alpha=0.85)
ax.set_xticks(x + 0.175)
ax.set_xticklabels(labels_r, rotation=20, fontsize=9)
ax.set_title('Características por perfil\n(normalizadas 0-1)', fontweight='bold')
ax.set_ylabel('Valor normalizado')
ax.legend(fontsize=8)
ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('/tmp/clustering_kmeans.png', dpi=150, bbox_inches='tight')
display(fig)
plt.close()
print("\n✓ Clustering completo")

In [0]:
# ── CELDA 6: Persistencia — demostración ────────────────────
# Criterio SI7006: persistencia de modelos y uso en producción

print("=" * 60)
print("PERSISTENCIA DE MODELOS — Vermont EWS")
print("Criterio SI7006: persistencia y publicación en APIs")
print("=" * 60)

VOLUME_DIR = f"{SILVER}/models"

# Verificar artefactos
print(f"\n── Artefactos en {VOLUME_DIR} ──")
try:
    files = dbutils.fs.ls(VOLUME_DIR)
    for f in files:
        print(f"  {f.name:45s} {f.size/1024:8.1f} KB")
except Exception as e:
    print(f"⚠ {e}")

# Demostración: cargar y predecir sin reentrenar
print("\n── Demostración: cargar modelo y predecir ──")
print("(sin reentrenar — solo cargando el artefacto persistido)")

try:
    rf_demo  = joblib.load(f"{VOLUME_DIR}/rf_classifier.joblib")
    imp_demo = joblib.load(f"{VOLUME_DIR}/imputer.joblib")
    le_demo  = joblib.load(f"{VOLUME_DIR}/label_encoder.joblib")
    with open(f"{VOLUME_DIR}/feature_list.json") as f:
        feats_demo = json.load(f)

    avail_demo = [f for f in feats_demo if f in df_predict.columns]
    X_demo     = imp_demo.transform(df_predict[avail_demo].head(5).fillna(0))
    probas_d   = rf_demo.predict_proba(X_demo)
    idx_c      = list(le_demo.classes_).index('critical')

    print(f"\n  {'ID':20s} {'P(critical)':>12s} {'Pred (0.30)':>14s}")
    print("  " + "-" * 50)
    for i, (_, row) in enumerate(df_predict.head(5).iterrows()):
        p_c  = probas_d[i][idx_c]
        pred = 'critical' if p_c >= 0.30 \
               else le_demo.classes_[np.argmax(probas_d[i])]
        print(f"  {str(row.get('student_id','—'))[:20]:20s} "
              f"{p_c:12.3f} {pred:>14s}")

    print(f"\n✓ Modelo operativo sin reentrenamiento")
    print(f"  Clases: {list(le_demo.classes_)}")
    print(f"  Features: {len(feats_demo)}")
    print(f"  Umbral: 0.30")

except Exception as e:
    print(f"⚠ Error: {e}")

In [0]:
# ── Guardar métricas EDA como JSON directo al Volume ────────
import json

metricas = {
    "notebook": "02b_eda_sql",
    "fecha": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"),
    "clasificacion": {
        "modelo": meta_clf['modelo'],
        "umbral": meta_clf['umbral_produccion'],
        "dataset_eval": "2025-26",
        "n_eval": len(df_predict),
        "f1_macro_cv": meta_clf['metricas_cv']['f1_macro'],
        "recall_critical_cv": meta_clf['metricas_cv']['recall_critical'],
        "auc_macro": round(np.mean(list(auc_scores.values())), 3)
                     if modelo_ok else None
    },
    "clustering": {
        "K_optimo_silhouette": int(K_OPTIMO),
        "K_usado_pipeline": 2,
        "silhouette_K2": round(
            silhouette_score(X_scaled, km_final.labels_), 3),
        "perfiles": nombres
    }
}

# Escribir directo al Volume — sin pasar por /tmp
metricas_json = json.dumps(metricas, indent=2, ensure_ascii=False)
spark.createDataFrame([(metricas_json,)], ["json_content"]) \
     .write.mode("overwrite") \
     .text(f"{SILVER}/eda_metricas_modelo")

print("✓ Métricas guardadas en Silver/eda_metricas_modelo")

In [0]:
import json

SILVER = "/Volumes/workspace/vermont/silver"

with open(f"{SILVER}/models/model_metadata.json") as f:
    meta_clf = json.load(f)
print("── Clasificador ──")
print(json.dumps(meta_clf, indent=2))

with open(f"{SILVER}/models/reg_metadata.json") as f:
    meta_reg = json.load(f)
print("\n── Regresor ──")
print(json.dumps(meta_reg, indent=2))

In [0]:
# ============================================================
# CELDA EXTRA — Tabla train / validación / test
# RF (cargado, sin reentrenar) vs Regresión Logística
# Solo lectura — no toca el pipeline
# ============================================================

import pandas as pd, numpy as np, json, joblib
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, recall_score

TRUSTED = "/Volumes/workspace/vermont/trusted"
SILVER  = "/Volumes/workspace/vermont/silver"

# ── Cargar datos ──
df_train = spark.read.parquet(f"{TRUSTED}/train_dataset_fe").toPandas()
df_test  = spark.read.parquet(f"{TRUSTED}/predict_dataset_fe").toPandas()

# ── Cargar artefactos del modelo de producción ──
rf      = joblib.load(f"{SILVER}/models/rf_classifier.joblib")
imp_rf  = joblib.load(f"{SILVER}/models/imputer.joblib")
le      = joblib.load(f"{SILVER}/models/label_encoder.joblib")
with open(f"{SILVER}/models/feature_list.json") as f:
    FEATURES = json.load(f)
with open(f"{SILVER}/models/model_metadata.json") as f:
    meta = json.load(f)

FEATURES = [c for c in FEATURES if c in df_train.columns and c in df_test.columns]
idx_crit = list(le.classes_).index("critical")

# ── Preparar X, y con el imputador del pipeline ──
X_train = imp_rf.transform(df_train[FEATURES])
X_test  = imp_rf.transform(df_test[FEATURES])
y_train = le.transform(df_train["risk_level"])
y_test  = le.transform(df_test["risk_level"])

def aplicar_umbral(model, X, umbral=0.30):
    """Predice aplicando umbral 0.30 sobre la clase crítica."""
    probas = model.predict_proba(X)
    preds = []
    for p in probas:
        if p[idx_crit] >= umbral:
            preds.append(idx_crit)
        else:
            resto = p.copy(); resto[idx_crit] = -1
            preds.append(int(np.argmax(resto)))
    return np.array(preds)

def metricas(model, X, y):
    pred = aplicar_umbral(model, X)
    return (round(accuracy_score(y, pred), 3),
            round(f1_score(y, pred, average="macro"), 3),
            round(recall_score(y, pred, labels=[idx_crit],
                               average="macro", zero_division=0), 3))

filas = []

# ════════════════════════════════════════════
# RANDOM FOREST — modelo de producción (NO se reentrena)
# ════════════════════════════════════════════
# Train: aplicar el modelo cargado sobre los datos de entrenamiento
tr = metricas(rf, X_train, y_train)
# Validación: tomar del metadata (5-Fold CV ya calculado en el pipeline)
val = (None,
       meta["metricas_cv"]["f1_macro"],
       meta["metricas_cv"]["recall_critical"])
# Test: aplicar el modelo cargado sobre 2025-26
te = metricas(rf, X_test, y_test)

filas.append(["Random Forest", "Train",      tr[0], tr[1], tr[2]])
filas.append(["Random Forest", "Validación", val[0], val[1], val[2]])
filas.append(["Random Forest", "Test",       te[0], te[1], te[2]])

# ════════════════════════════════════════════
# REGRESIÓN LOGÍSTICA — sí se entrena (no está persistida)
# ════════════════════════════════════════════
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
lr  = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)

# Validación con CV
cv = cross_validate(lr, X_train, y_train, cv=skf, scoring=["accuracy", "f1_macro"])
lr_val_acc = round(cv["test_accuracy"].mean(), 3)
lr_val_f1  = round(cv["test_f1_macro"].mean(), 3)

rec_cv = []
for tr_idx, va_idx in skf.split(X_train, y_train):
    m = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
    m.fit(X_train[tr_idx], y_train[tr_idx])
    pred_va = aplicar_umbral(m, X_train[va_idx])
    rec_cv.append(recall_score(y_train[va_idx], pred_va, labels=[idx_crit],
                               average="macro", zero_division=0))
lr_val_rec = round(np.mean(rec_cv), 3)

# Entrenar en todo el train para train y test
lr.fit(X_train, y_train)
lr_tr = metricas(lr, X_train, y_train)
lr_te = metricas(lr, X_test, y_test)

filas.append(["Regresión Logística", "Train",      lr_tr[0], lr_tr[1], lr_tr[2]])
filas.append(["Regresión Logística", "Validación", lr_val_acc, lr_val_f1, lr_val_rec])
filas.append(["Regresión Logística", "Test",       lr_te[0], lr_te[1], lr_te[2]])

# ── Tabla final ──
tabla = pd.DataFrame(filas, columns=["Modelo", "Conjunto",
                                     "Accuracy", "F1 macro", "Recall critical"])
print("="*65)
print("TABLA TRAIN / VALIDACIÓN / TEST — umbral 0.30")
print("="*65)
print(tabla.to_string(index=False))
print("\nNota: RF no se reentrena (modelo de producción cargado).")
print("Validación RF tomada de model_metadata.json (5-Fold CV del pipeline).")
print("Regresión Logística sí se entrena (no está persistida).")